# How to Choose a Linear Solver
> **Set up**
>
> To run this notebook, first install the Julia kernel for Jupyter Notebooks using [IJulia](https://julialang.github.io/IJulia.jl/stable/manual/installation/), then [create an environment](https://pkgdocs.julialang.org/v1/environments/) for this tutorial with the packages listed with `using <PackageName>` further down.
>
> This tutorial has demonstrated compatibility with these package versions. If you run into any errors, first check your package versions for consistency using `Pkg.status()`.
>
 > ```
 > Status `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl/docs/Project.toml`
 >   [a93c6f00] DataFrames v1.8.2
 >   [864edb3b] DataStructures v0.19.6
 >   [e30172f5] Documenter v1.17.0
 >   [d12716ef] DocumenterInterLinks v1.1.0
 >   [98b081ad] Literate v2.21.0
 >   [bed98974] PowerNetworkMatrices v0.24.3 `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl`
 >   [f00506e0] PowerSystemCaseBuilder v2.5.0
 >   [bcd98974] PowerSystems v5.12.0
 >   [08abe8d2] PrettyTables v3.4.3
 > 
 > ```



This guide helps you select the appropriate linear solver for your network matrix computations.

## Available Solvers

Pass the solver name as the `linear_solver` keyword to any matrix constructor
(`PTDF`, `LODF`, `ABA_Matrix`, …). `PowerNetworkMatrices.jl` supports four:

 1. **`"KLU"`** - sparse
    [KLU](https://github.com/DrTimothyAldenDavis/SuiteSparse) factorization
    Always available (built-in `KLUWrapper` submodule); the default off Apple hardware.
 2. **`"AppleAccelerateLU"`** - sparse LU via Apple's
    [Accelerate sparse solvers](https://developer.apple.com/documentation/accelerate/sparse_solvers).
    Always compiled in (built-in `AccelerateWrapper` submodule) but
    runtime-gated to macOS 15.5+ on Apple hardware, where it is the default.
 3. **`"MKLPardiso"`** - Intel's
    [oneMKL PARDISO](https://www.intel.com/content/www/us/en/docs/onemkl/developer-reference-c/),
    wrapped by [`Pardiso.jl`](https://github.com/JuliaSparse/Pardiso.jl). A
    weak-dependency package extension: only loaded once you also add and import
    `Pardiso.jl`.
 4. **`"Dense"`** - dense
    [LU](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.lu)
    from Julia's LinearAlgebra stdlib, for small or debugging cases.

The default is platform-dependent: `AppleAccelerateLU` on macOS 15.5+ (Apple
hardware), `KLU` everywhere else. KLU and Apple Accelerate are always-present
submodules — only MKL Pardiso is an optional extension.

The examples below use a small test system loaded with `PowerSystemCaseBuilder`:

In [ ]:
using PowerNetworkMatrices
import PowerNetworkMatrices as PNM
import PowerSystemCaseBuilder as PSB

sys = PSB.build_system(PSB.PSITestSystems, "c_sys5");

## Choosing the Right Solver

### Use KLU When:

  - Working with typical power systems (most cases)
  - System size is medium to large (> 100 buses)
  - You want good performance without special dependencies
  - Running on any platform (Linux, macOS, Windows)

Use `PTDF` with the KLU solver (the default off Apple hardware):

In [ ]:
ptdf_matrix = PTDF(sys)  # platform default

or explicitly:

In [ ]:
ptdf_matrix = PTDF(sys; linear_solver = "KLU");

### Use Apple Accelerate When:

  - Running on Apple-silicon macOS 15.5 or newer
  - You want the platform-tuned sparse LU (it is the default there)

Select it explicitly with:

```julia
ptdf_matrix = PTDF(sys; linear_solver = "AppleAccelerateLU");
```

### Use Dense When:

  - System is very small (< 30 buses)
  - You're debugging or validating results
  - Matrix operations are simple and small-scale

Specify the Dense solver explicitly:

In [ ]:
ptdf_matrix = PTDF(sys; linear_solver = "Dense");

### Use MKLPardiso When:

  - You have Intel processors
  - Running on Linux or Windows (not available on Apple silicon)
  - Maximum performance is critical
  - Working with very large systems (> 1000 buses)

MKLPardiso lives in a weak-dependency package extension, so first add and
import `Pardiso.jl` to load it, then request the solver:

```julia
using Pardiso   # loads the MKLPardisoExt extension
ptdf_matrix = PTDF(sys; linear_solver = "MKLPardiso")
```

## Performance Considerations

### System Size

| Buses   | Recommended Solver |
|:------- |:------------------ |
| < 30    | Dense or KLU       |
| 30-1000 | KLU                |
| > 1000  | KLU or MKLPardiso  |

### Platform Availability

| Solver            | Linux | Windows | macOS             |
|:----------------- |:----- |:------- |:----------------- |
| KLU               | ✓     | ✓       | ✓                 |
| AppleAccelerateLU | ✗     | ✗       | ✓ (Apple, 15.5+)  |
| Dense             | ✓     | ✓       | ✓                 |
| MKLPardiso        | ✓     | ✓       | ✗                 |

`AppleAccelerateLU` needs no extra package — it is built in. Only
`MKLPardiso` requires installing and importing `Pardiso.jl`.

### Singular `ABA`: prefer KLU

The two default solvers differ in how they treat a singular `ABA` matrix.
`AppleAccelerateLU` silently factorizes a singular matrix and returns garbage,
whereas KLU raises. Prefer `"KLU"` whenever the `ABA` matrix may be singular — for
example a full outage that isolates a bus, such as a 3-winding transformer's
zero-injection star bus:

```julia
ptdf = PTDF(sys; linear_solver = "KLU")   # safe when singularity is possible
```

### Persisting a preferred backend

The active sparse backend can be pinned across Julia sessions with `Preferences.jl`
via the (non-exported) helpers in `src/linalg_settings.jl` —
`PNM.set_linalg_backend_preference` / `PNM.get_linalg_backend_preference`,
`PNM.set_linalg_backend_check` / `PNM.get_linalg_backend_check`, and
`PNM.check_linalg_backend` (reports the active BLAS/LAPACK library and whether the
requested backend is loaded).

## Troubleshooting

### MKLPardiso Not Available

If you get an error when using MKLPardiso:

 1. Confirm you have added and imported `Pardiso.jl` (the extension only
    loads once `Pardiso` is available)
 2. Verify you're on Linux or Windows (not macOS)
 3. Check that you have Intel processors

Fall back to KLU if MKLPardiso is unavailable — it is always present.

## Related Topics

  - How to Build Multiple Matrices Without Repeating Work - use these solvers to build matrices
  - Computational Considerations - the reasoning behind the platform defaults
  - The `AutoTolerance` docstring - the orthogonal `tol` sparsification setting